In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# =====================================================
# STEP 1: CREATE SALES REPRESENTATIVES
# =====================================================

# Define performance archetypes (creates realistic distribution)
def assign_performance_tier():
    rand = np.random.random()
    if rand < 0.15:
        return "Top Performer"  # 15% - exceeds quota consistently
    elif rand < 0.55:
        return "Solid Performer"  # 40% - meets quota
    elif rand < 0.85:
        return "Underperformer"  # 30% - misses quota
    else:
        return "At-Risk"  # 15% - significantly underperforming

# Generate 40 sales reps
reps_data = []
first_names = ['Alex', 'Maria', 'James', 'Priya', 'David', 'Sofia', 'Michael', 'Aisha', 
               'Robert', 'Fatima', 'John', 'Zara', 'Chris', 'Layla', 'Daniel', 'Nadia',
               'Ahmed', 'Emma', 'Hassan', 'Olivia', 'Karan', 'Sophie', 'Ravi', 'Lucia',
               'Omar', 'Anna', 'Yusuf', 'Isabel', 'Faisal', 'Diana', 'Sanjay', 'Elena',
               'Tariq', 'Grace', 'Nikhil', 'Sara', 'Kunal', 'Alice', 'Rohan', 'Julia']

last_names = ['Chen', 'Patel', 'Smith', 'Kumar', 'Wilson', 'Rodriguez', 'Nguyen', 'Khan',
              'Martinez', 'Ali', 'Brown', 'Sharma', 'Johnson', 'Ahmed', 'Garcia', 'Rahman',
              'Williams', 'Jones', 'Lee', 'Kim', 'Singh', 'Park', 'Zhang', 'Wang',
              'Lopez', 'Anderson', 'Taylor', 'Thomas', 'Moore', 'Jackson', 'White', 'Harris',
              'Martin', 'Thompson', 'Robinson', 'Clark', 'Lewis', 'Walker', 'Hall', 'Young']

territories = ['North America', 'EMEA', 'APAC', 'LATAM']
segments = ['SMB', 'Mid-Market', 'Enterprise']

for i in range(40):
    rep = {
        'rep_id': f'REP{str(i+1).zfill(3)}',
        'first_name': first_names[i],
        'last_name': last_names[i],
        'full_name': f"{first_names[i]} {last_names[i]}",
        'territory': np.random.choice(territories, p=[0.35, 0.30, 0.20, 0.15]),
        'primary_segment': np.random.choice(segments, p=[0.40, 0.40, 0.20]),
        'performance_tier': assign_performance_tier(),
        'hire_date': datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 730)),
        'annual_quota': np.random.choice([600000, 750000, 900000, 1200000], p=[0.3, 0.35, 0.25, 0.10]),
    }
    reps_data.append(rep)

reps_df = pd.DataFrame(reps_data)

# Introduce a "problem territory" - APAC has structural issues
apac_reps = reps_df[reps_df['territory'] == 'APAC'].index
for idx in apac_reps:
    if np.random.random() < 0.7:  # 70% of APAC reps are underperforming
        reps_df.loc[idx, 'performance_tier'] = np.random.choice(['Underperformer', 'At-Risk'], p=[0.6, 0.4])

print(f"Generated {len(reps_df)} sales representatives")
print(reps_df['performance_tier'].value_counts())
print(reps_df['territory'].value_counts())

# =====================================================
# STEP 2: GENERATE MONTHLY PERFORMANCE DATA
# =====================================================

def generate_monthly_metrics(rep, month_date, months_tenure):
    """Generate realistic monthly performance metrics for a rep"""
    
    # Base performance multipliers by tier
    tier_multipliers = {
        'Top Performer': (1.10, 1.35),  # 110-135% of quota
        'Solid Performer': (0.85, 1.10),  # 85-110% of quota
        'Underperformer': (0.55, 0.85),  # 55-85% of quota
        'At-Risk': (0.20, 0.60)  # 20-60% of quota
    }
    
    tier = rep['performance_tier']
    multiplier_range = tier_multipliers[tier]
    
    # New hire ramp effect (first 6 months significantly reduced)
    if months_tenure < 6:
        ramp_factor = 0.3 + (months_tenure * 0.11)  # 30% to 96% over 6 months
    else:
        ramp_factor = 1.0
    
    # Monthly quota (annual / 12)
    monthly_quota = rep['annual_quota'] / 12
    
    # Actual monthly attainment
    base_attainment = np.random.uniform(multiplier_range[0], multiplier_range[1])
    monthly_attainment = base_attainment * ramp_factor
    monthly_revenue = monthly_quota * monthly_attainment
    
    # Seasonal patterns (Q4 stronger, Q1 weaker)
    quarter = (month_date.month - 1) // 3 + 1
    seasonal_factor = {1: 0.85, 2: 1.0, 3: 1.05, 4: 1.20}
    monthly_revenue *= seasonal_factor[quarter]
    
    # Activity metrics correlated with performance
    calls_base = 45 if tier == 'Top Performer' else 30 if tier == 'Solid Performer' else 20
    calls = int(np.random.normal(calls_base, 8))
    
    meetings_base = 20 if tier == 'Top Performer' else 12 if tier == 'Solid Performer' else 7
    meetings = int(np.random.normal(meetings_base, 3))
    
    demos_base = 8 if tier == 'Top Performer' else 5 if tier == 'Solid Performer' else 3
    demos = int(np.random.normal(demos_base, 1))
    
    # Deal metrics
    deals_closed = int(np.random.normal(4 if tier == 'Top Performer' else 2.5 if tier == 'Solid Performer' else 1.5, 1))
    deals_closed = max(0, deals_closed)
    
    avg_deal_size = monthly_revenue / max(deals_closed, 1)
    
    # Pipeline metrics
    pipeline_coverage = 3.5 if tier == 'Top Performer' else 2.8 if tier == 'Solid Performer' else 2.0
    pipeline_coverage += np.random.uniform(-0.5, 0.5)
    pipeline_value = monthly_quota * pipeline_coverage
    
    # Win rate
    win_rate_base = 0.35 if tier == 'Top Performer' else 0.25 if tier == 'Solid Performer' else 0.15
    win_rate = win_rate_base + np.random.uniform(-0.05, 0.05)
    
    # Deal cycle length
    cycle_length_base = 45 if tier == 'Top Performer' else 60 if tier == 'Solid Performer' else 85
    avg_cycle_days = int(np.random.normal(cycle_length_base, 10))
    
    return {
        'rep_id': rep['rep_id'],
        'month_date': month_date,
        'year': month_date.year,
        'month': month_date.month,
        'quarter': quarter,
        'monthly_quota': round(monthly_quota, 2),
        'monthly_revenue': round(max(0, monthly_revenue), 2),
        'quota_attainment_pct': round((monthly_revenue / monthly_quota) * 100, 2),
        'calls_made': max(0, calls),
        'meetings_held': max(0, meetings),
        'demos_completed': max(0, demos),
        'deals_closed': deals_closed,
        'avg_deal_size': round(avg_deal_size, 2),
        'pipeline_value': round(pipeline_value, 2),
        'pipeline_coverage_ratio': round(pipeline_coverage, 2),
        'win_rate': round(win_rate, 3),
        'avg_cycle_days': avg_cycle_days,
        'months_tenure': months_tenure
    }

# Generate 24 months of data for each rep
monthly_data = []
end_date = datetime(2025, 12, 31)
start_date = datetime(2024, 1, 1)

for _, rep in reps_df.iterrows():
    hire_date = rep['hire_date']
    current_date = max(start_date, hire_date.replace(day=1))
    
    while current_date <= end_date:
        months_tenure = (current_date.year - hire_date.year) * 12 + (current_date.month - hire_date.month)
        if months_tenure >= 0:
            monthly_metric = generate_monthly_metrics(rep, current_date, months_tenure)
            monthly_data.append(monthly_metric)
        
        # Move to next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

monthly_df = pd.DataFrame(monthly_data)
print(f"\nGenerated {len(monthly_df)} monthly performance records")

# =====================================================
# STEP 3: GENERATE DEAL-LEVEL DATA
# =====================================================

deal_data = []
deal_id_counter = 1

for _, rep in reps_df.iterrows():
    # Number of deals varies by tier
    tier = rep['performance_tier']
    num_deals = {
        'Top Performer': np.random.randint(60, 90),
        'Solid Performer': np.random.randint(40, 60),
        'Underperformer': np.random.randint(20, 40),
        'At-Risk': np.random.randint(10, 25)
    }[tier]
    
    for _ in range(num_deals):
        # Deal characteristics
        segment = rep['primary_segment']
        segment_deal_size = {
            'SMB': (5000, 25000),
            'Mid-Market': (25000, 100000),
            'Enterprise': (100000, 500000)
        }
        deal_size = np.random.randint(*segment_deal_size[segment])
        
        # Deal date within 2024-2025
        deal_date = datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 730))
        
        # Win/loss with tier-based probability
        win_prob = {
            'Top Performer': 0.65,
            'Solid Performer': 0.45,
            'Underperformer': 0.28,
            'At-Risk': 0.15
        }[tier]
        
        deal_status = 'Won' if np.random.random() < win_prob else 'Lost'
        
        # Loss reasons (if lost)
        if deal_status == 'Lost':
            loss_reason = np.random.choice([
                'Price', 'Competitor', 'No Decision', 'Poor Fit', 
                'Timing', 'Missing Feature', 'Budget Cut'
            ], p=[0.25, 0.20, 0.15, 0.10, 0.15, 0.10, 0.05])
        else:
            loss_reason = None
        
        # Cycle length
        cycle_length_base = {
            'SMB': 30, 'Mid-Market': 60, 'Enterprise': 120
        }[segment]
        cycle_days = int(np.random.normal(cycle_length_base, cycle_length_base * 0.3))
        cycle_days = max(7, cycle_days)
        
        # Discount
        discount_pct = np.random.choice([0, 5, 10, 15, 20, 25, 30], 
                                         p=[0.30, 0.25, 0.20, 0.10, 0.08, 0.05, 0.02])
        
        deal_data.append({
            'deal_id': f'DEAL{str(deal_id_counter).zfill(5)}',
            'rep_id': rep['rep_id'],
            'deal_date': deal_date,
            'customer_segment': segment,
            'territory': rep['territory'],
            'deal_size_usd': deal_size,
            'discount_pct': discount_pct,
            'final_deal_value': round(deal_size * (1 - discount_pct/100), 2),
            'cycle_days': cycle_days,
            'deal_status': deal_status,
            'loss_reason': loss_reason
        })
        deal_id_counter += 1

deals_df = pd.DataFrame(deal_data)
print(f"\nGenerated {len(deals_df)} deal records")
print(deals_df['deal_status'].value_counts())

# =====================================================
# STEP 4: SAVE ALL DATA
# =====================================================

reps_df.to_csv('../data/sales_reps.csv', index=False)
monthly_df.to_csv('../data/monthly_performance.csv', index=False)
deals_df.to_csv('../data/deals.csv', index=False)

print("\n=== DATA GENERATION COMPLETE ===")
print(f"Sales Reps: {len(reps_df)} records")
print(f"Monthly Performance: {len(monthly_df)} records")
print(f"Deals: {len(deals_df)} records")
print("\nFiles saved to data/ folder")

Generated 40 sales representatives
performance_tier
Solid Performer    18
Underperformer     12
At-Risk             6
Top Performer       4
Name: count, dtype: int64
territory
North America    14
EMEA             13
APAC              9
LATAM             4
Name: count, dtype: int64

Generated 820 monthly performance records

Generated 1635 deal records
deal_status
Lost    886
Won     749
Name: count, dtype: int64

=== DATA GENERATION COMPLETE ===
Sales Reps: 40 records
Monthly Performance: 820 records
Deals: 1635 records

Files saved to data/ folder
